# Clustering Narratives

#### INPUT  
- narrative_extractions_normalized.csv

#### WHAT THIS NOTEBOOK DOES
- loads the normalized extractions produced by notebook 04_normalize.ipynb
- builds a structural feature matrix and embeds dominant narratives and slot labels
- combines the feature blocks and reduces them with UMAP
- clusters at three levels with HDBSCAN: macro (structural), meso (semantic sub-types), micro (perspective)
- profiles the resulting clusters and validates the clustering

#### OUTPUTS 
- cluster_assignments.csv (+ cluster_assignments_macro / _meso / _micro.csv) --> per-narrative cluster labels
- feature_matrix.npy / feature_ids.npy / feature_columns.npy --> structural feature matrix and its id/column keys
- embeddings.npy / embedding_ids.npy --> dominant-narrative embeddings and their ids
- slot_embeddings.npy / slot_embedding_ids.npy --> slot-label embeddings and their ids
- combined_matrix.npy / reduced_embeddings.npy --> combined feature blocks and the UMAP-reduced matrix


In [ ]:
# imports

from config import cluster_config as cfg
from lib.cluster_lib import (
    load_extractions,
    build_feature_matrix, save_feature_matrix, load_feature_matrix, feature_matrix_exists,
    load_model, embed_texts, save_embeddings, load_embeddings, embeddings_exist,
    build_slot_embeddings, save_slot_embeddings, load_slot_embeddings, slot_embeddings_exist,
    combine_features, save_combined, load_combined, combined_exists,
    reduce_for_clustering, save_reduced, load_reduced, reduced_exists,
    cluster, cluster_within, proportional_min_cluster_size,
    join_extractions, profile_noise,
    tune_macro, tune_within, validate_clustering,
)
import numpy as np
import pandas as pd
from tqdm import tqdm


In [2]:
# load data

extractions = load_extractions(cfg.EXTRACTIONS_CSV)

print(f"Extraction rows:                {len(extractions):,}")
print(f"Unique articles with narrative: {extractions['id'].nunique():,}")
extractions.head()

Extraction rows:                417,458
Unique articles with narrative: 44,876


,id,narrative_present,dominant_narrative,slot,filler_rank,label,filler_type,specific_type,category,domain,uncertain,uncertainty_note,pubtime,medium_code,medium_name,char_count
0,58672106,True,Russian Foreign Minister Lavrov pursues intern...,subject,0,Sergei Lavrov / Russian government,actor,foreign_executive,foreign_actor,international_diplomatic,False,NaN,2025-10-28 20:27:22+00:00,ZWAO,20 minuten online,2744
1,58672106,True,Russian Foreign Minister Lavrov pursues intern...,object,0,International acceptance of Russia as security...,abstract_force,policy_instrument,deployed_instrument,international_security,False,NaN,2025-10-28 20:27:22+00:00,ZWAO,20 minuten online,2744
2,58672106,True,Russian Foreign Minister Lavrov pursues intern...,sender,0,Russian state interest in restoring geopolitic...,ideology,communitarian,cultural_axis,international_diplomatic,True,Lavrov's framing appeals to sovereignty and re...,2025-10-28 20:27:22+00:00,ZWAO,20 minuten online,2744
3,58672106,True,Russian Foreign Minister Lavrov pursues intern...,receiver,0,EU and NATO member states,actor,foreign_executive,foreign_actor,international_security,False,NaN,2025-10-28 20:27:22+00:00,ZWAO,20 minuten online,2744
4,58672106,True,Russian Foreign Minister Lavrov pursues intern...,opponent,0,Russia's documented history of treaty violatio...,abstract_force,geopolitical,structural_condition,international_security,False,NaN,2025-10-28 20:27:22+00:00,ZWAO,20 minuten online,2744


In [3]:
# build or load feature matrix (structural: category, specific_type, domain)

if feature_matrix_exists(cfg.FEATURE_MATRIX_NPY, cfg.FEATURE_IDS):
    print("Loading cached feature matrix...")
    matrix, ids, columns = load_feature_matrix(cfg.FEATURE_MATRIX_NPY, cfg.FEATURE_IDS, cfg.FEATURE_COLUMNS_NPY)
else:
    print("Building feature matrix...")
    matrix, ids, columns = build_feature_matrix(extractions)
    save_feature_matrix(matrix, ids, columns, cfg.FEATURE_MATRIX_NPY, cfg.FEATURE_IDS, cfg.FEATURE_COLUMNS_NPY)

print(f"Feature matrix shape: {matrix.shape}")

Loading cached feature matrix...
Feature matrix shape: (44876, 574)


In [4]:
# build or load dominant narrative embeddings

if embeddings_exist(cfg.EMBEDDINGS_NPY, cfg.EMBEDDING_IDS):
    print("Loading cached embeddings...")
    embeddings, embedding_ids = load_embeddings(cfg.EMBEDDINGS_NPY, cfg.EMBEDDING_IDS)
else:
    print("Embedding dominant narratives (~20-40 min)...")
    narratives    = extractions[["id", "dominant_narrative"]].drop_duplicates("id").dropna(subset=["dominant_narrative"])
    model         = load_model(cfg.EMBEDDING_MODEL)
    embeddings    = embed_texts(model, narratives["dominant_narrative"].tolist(), cfg.EMBEDDING_BATCH_SIZE)
    embedding_ids = narratives["id"].values
    save_embeddings(embeddings, embedding_ids, cfg.EMBEDDINGS_NPY, cfg.EMBEDDING_IDS)

print(f"Embeddings shape: {embeddings.shape}")

Loading cached embeddings...
Embeddings shape: (44876, 1024)


In [5]:
# build or load slot label embeddings

if slot_embeddings_exist(cfg.SLOT_EMBEDDINGS_NPY, cfg.SLOT_EMBEDDING_IDS):
    print("Loading cached slot embeddings...")
    slot_embeddings, slot_embedding_ids = load_slot_embeddings(cfg.SLOT_EMBEDDINGS_NPY, cfg.SLOT_EMBEDDING_IDS)
else:
    print("Building slot label embeddings...")
    if "model" not in dir():
        model = load_model(cfg.EMBEDDING_MODEL)
    slot_embeddings, slot_embedding_ids = build_slot_embeddings(extractions, model, cfg.EMBEDDING_BATCH_SIZE)
    save_slot_embeddings(slot_embeddings, slot_embedding_ids, cfg.SLOT_EMBEDDINGS_NPY, cfg.SLOT_EMBEDDING_IDS)

print(f"Slot embeddings shape: {slot_embeddings.shape}")

Loading cached slot embeddings...
Slot embeddings shape: (44876, 6144)


## Macro Clustering

In [6]:
# combine features and run UMAP (macro level)
# these are cached — only runs once

if combined_exists(cfg.COMBINED_MATRIX_NPY):
    print("Loading cached combined matrix...")
    combined = load_combined(cfg.COMBINED_MATRIX_NPY)
else:
    print("Combining features (macro weights)...")
    combined, _ = combine_features(
        matrix, embeddings, ids, embedding_ids,
        cfg.MACRO_SEMANTIC_WEIGHT,
        slot_embeddings=slot_embeddings,
        slot_embedding_ids=slot_embedding_ids,
        slot_weight=cfg.MACRO_SLOT_WEIGHT,
    )
    save_combined(combined, cfg.COMBINED_MATRIX_NPY)

if reduced_exists(cfg.REDUCED_NPY):
    print("Loading cached reduced matrix...")
    reduced = load_reduced(cfg.REDUCED_NPY)
else:
    print("Running macro UMAP (~5-10 min)...")
    reduced = reduce_for_clustering(
        combined, cfg.UMAP_N_COMPONENTS, cfg.MACRO_N_NEIGHBORS, cfg.UMAP_MIN_DIST, cfg.UMAP_METRIC
    )
    save_reduced(reduced, cfg.REDUCED_NPY)

print(f"Combined: {combined.shape}  Reduced: {reduced.shape}")

Loading cached combined matrix...
Running macro UMAP (~5-10 min)...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Combined: (44876, 1598)  Reduced: (44876, 50)


In [7]:
# [TUNE MACRO] grid search over n_neighbors, min_cluster_size, min_samples
# Takes combined matrix so n_neighbors can be varied (UMAP re-runs per n_neighbors value).
# Scores on n_clusters target range and noise rate only — NOT coherence.
# Macro clusters are broad thematic umbrellas where no single actor dominates;
# top10_cumul will always be low at this level and is not a valid criterion.
# Run to find good parameters, then update config and delete data/reduced_embeddings.npy
# before running the RUN cell so UMAP uses the new n_neighbors.
# Skip if parameters are already confirmed.

macro_tune_df = tune_macro(
    combined=combined,
    n_neighbors_values=cfg.TUNE_MACRO_N_NEIGHBORS,
    min_cluster_sizes=cfg.TUNE_MACRO_MIN_CLUSTER_SIZES,
    min_samples_values=cfg.TUNE_MACRO_MIN_SAMPLES,
    umap_n_components=cfg.UMAP_N_COMPONENTS,
    umap_min_dist=cfg.UMAP_MIN_DIST,
    umap_metric=cfg.UMAP_METRIC,
    hdbscan_metric=cfg.HDBSCAN_METRIC,
    target_n_min=cfg.TUNE_MACRO_TARGET_N_MIN,
    target_n_max=cfg.TUNE_MACRO_TARGET_N_MAX,
    target_noise_max=cfg.COHERENCE_TARGET_NOISE_MAX,
)

print(macro_tune_df.to_string(index=False))

rec = macro_tune_df.iloc[0]
print(f"\nRecommended config:")
print(f"  MACRO_N_NEIGHBORS      = {int(rec['n_neighbors'])}")
print(f"  MACRO_MIN_CLUSTER_SIZE = {int(rec['min_cluster_size'])}")
print(f"  MACRO_MIN_SAMPLES      = {int(rec['min_samples'])}")
print(f"  → {int(rec['n_clusters'])} clusters, {rec['noise_rate']:.1%} noise")
print(f"\nIf n_neighbors changed: delete data/reduced_embeddings.npy before running RUN cell.")

  UMAP n_neighbors=100...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  UMAP n_neighbors=150...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  UMAP n_neighbors=200...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  UMAP n_neighbors=300...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


 RANK  n_neighbors  min_cluster_size  min_samples  n_clusters  noise_rate  in_target  noise_pen  score
    1          300               300           25          11       0.058        1.0        0.0  0.942
    2          300               100           25          14       0.063        1.0        0.0  0.937
    3          200               300           50          13       0.068        1.0        0.0  0.932
    4          200               200           50          16       0.074        1.0        0.0  0.926
    5          200               150           50          17       0.083        1.0        0.0  0.917
    6          300               200           75          12       0.084        1.0        0.0  0.916
    7          300               200           25          15       0.096        1.0        0.0  0.904
    8          300               150           25          16       0.105        1.0        0.0  0.895
    9          300               300           75          13       0.107

In [7]:
# [RUN MACRO] cluster with config parameters and save

import importlib
from config import cluster_config as cfg
importlib.reload(cfg)

# sanity check — warn if cached reduced matrix may be stale
if reduced_exists(cfg.REDUCED_NPY):
    print("NOTE: using cached reduced matrix — if MACRO_N_NEIGHBORS changed, "
          "delete data/reduced_embeddings.npy and re-run the combine+UMAP cell first.")

macro_labels = cluster(
    reduced,
    min_cluster_size=cfg.MACRO_MIN_CLUSTER_SIZE,
    min_samples=cfg.MACRO_MIN_SAMPLES,
    metric=cfg.HDBSCAN_METRIC,
)

macro_assignments = pd.DataFrame({
    "id":            ids.astype(str),
    "macro_cluster": np.where(macro_labels >= 0, macro_labels.astype(float), np.nan),
})
macro_assignments.to_csv(cfg.MACRO_ASSIGNMENTS_CSV, index=False)

n_macro    = len(set(macro_labels[macro_labels >= 0]))
noise_rate = (macro_labels == -1).mean()
print(f"Macro clusters: {n_macro}")
print(f"Noise rate:     {noise_rate:.1%}")
print(f"Saved → {cfg.MACRO_ASSIGNMENTS_CSV}")
print(macro_assignments["macro_cluster"].value_counts().sort_index())

NOTE: using cached reduced matrix — if MACRO_N_NEIGHBORS changed, delete data/reduced_embeddings.npy and re-run the combine+UMAP cell first.
Macro clusters: 17
Noise rate:     8.3%
Saved → data/cluster_assignments_macro.csv
macro_cluster
0.0     4087
1.0     5842
2.0     1156
3.0     4589
4.0     1144
5.0     1049
6.0      208
7.0     6086
8.0      222
9.0     1129
10.0     190
11.0     405
12.0     760
13.0    9580
14.0    1122
15.0     282
16.0    3314
Name: count, dtype: int64


In [9]:
# [VALIDATE MACRO] coherence check — flags catch-all and tiny clusters

macro_val = validate_clustering(
    assignments=macro_assignments,
    cluster_col="macro_cluster",
    extractions=extractions,
    catchall_threshold=cfg.COHERENCE_CATCHALL_THRESHOLD,
    min_size_warning=cfg.COHERENCE_MIN_SIZE_WARNING,
    target_noise_max=cfg.COHERENCE_TARGET_NOISE_MAX,
    target_coherence_min=cfg.COHERENCE_TARGET_COHERENCE,
    target_catchall_max=cfg.COHERENCE_TARGET_CATCHALL_MAX,
    check_coherence=False,
)
macro_val


VALIDATION — MACRO_CLUSTER
  Clusters:        17
  Noise rate:      8.3%  (target < 25%)
  Avg coherence:   22.1%  (top10_cumul, target > 30%)
  Catch-all rate:  76%  (target < 40%)
  Tiny clusters:   0  (< 25 articles)

  VERDICT: ✓ PASS
  (coherence not evaluated — macro clusters are intentionally actor-diverse)

  Flagged clusters (worst first):
 cluster  n_articles  top10_cumul     flags
       2        1156         0.09 CATCH-ALL
       7        6086         0.09 CATCH-ALL
      14        1122         0.09 CATCH-ALL
       3        4589         0.12 CATCH-ALL
      11         405         0.13 CATCH-ALL
      12         760         0.13 CATCH-ALL
      13        9580         0.17 CATCH-ALL
       5        1049         0.17 CATCH-ALL
       8         222         0.21 CATCH-ALL
      16        3314         0.21 CATCH-ALL
       4        1144         0.24 CATCH-ALL
       9        1129         0.27 CATCH-ALL
      15         282         0.29 CATCH-ALL


,cluster,n_articles,n_uniq_subj,top1_pct,top10_cumul,flags
2,2,1156,1161,0.03,0.09,CATCH-ALL
7,7,6086,5115,0.02,0.09,CATCH-ALL
14,14,1122,1156,0.03,0.09,CATCH-ALL
3,3,4589,3643,0.02,0.12,CATCH-ALL
11,11,405,384,0.02,0.13,CATCH-ALL
12,12,760,764,0.02,0.13,CATCH-ALL
13,13,9580,6329,0.05,0.17,CATCH-ALL
5,5,1049,901,0.05,0.17,CATCH-ALL
8,8,222,205,0.04,0.21,CATCH-ALL
16,16,3314,2237,0.09,0.21,CATCH-ALL


## Meso Clustering

In [11]:
# [TUNE MESO] grid search over semantic_weight, n_neighbors_fraction, mcs_fraction, min_samples
# Fixed: slot_weight=0.0 (confirmed), umap_n_components, umap_min_dist, umap_metric, mcs_min, mcs_max
# UMAP runs once per (semantic_weight, n_neighbors_fraction, parent_cluster)
# HDBSCAN params vary on cached reduced space — no UMAP re-run per HDBSCAN combo
# Skip if parameters are already confirmed

meso_tune_df = tune_within(
    matrix=matrix, embeddings=embeddings, ids=ids, embedding_ids=embedding_ids,
    slot_embeddings=slot_embeddings, slot_embedding_ids=slot_embedding_ids,
    extractions=extractions,
    parent_assignments=macro_assignments,
    parent_col="macro_cluster",
    semantic_weights=cfg.TUNE_MESO_SEMANTIC_WEIGHTS,
    slot_weights=cfg.TUNE_MESO_SLOT_WEIGHTS,
    n_neighbors_fractions=cfg.TUNE_MESO_N_NEIGHBORS_FRACS,
    mcs_fractions=cfg.TUNE_MESO_MCS_FRACTIONS,
    mcs_min=cfg.MESO_MIN_CLUSTER_SIZE_MIN,
    mcs_max=cfg.MESO_MIN_CLUSTER_SIZE_MAX,
    min_samples_values=cfg.TUNE_MESO_MIN_SAMPLES,
    n_neighbors_min=cfg.MESO_N_NEIGHBORS_MIN,
    n_neighbors_max=cfg.MESO_N_NEIGHBORS_MAX,
    umap_n_components=cfg.UMAP_N_COMPONENTS,
    umap_min_dist=cfg.UMAP_MIN_DIST,
    umap_metric=cfg.UMAP_METRIC,
    hdbscan_metric=cfg.HDBSCAN_METRIC,
    test_parent_ids=cfg.TUNE_MESO_TEST_PARENT_IDS,
    target_noise_max=cfg.COHERENCE_TARGET_NOISE_MAX,
    catchall_threshold=cfg.COHERENCE_CATCHALL_THRESHOLD,
)

print(meso_tune_df.head(20).to_string(index=False))

rec = meso_tune_df.iloc[0]
print(f"\nRecommended config:")
print(f"  MESO_SEMANTIC_WEIGHT           = {rec['semantic_weight']}")
print(f"  MESO_N_NEIGHBORS_FRACTION      = {rec['n_neighbors_frac']}")
print(f"  MESO_MIN_CLUSTER_SIZE_FRACTION = {rec['mcs_fraction']}")
print(f"  MESO_MIN_SAMPLES               = {int(rec['min_samples'])}")

  Tuning on 3 parent clusters: [0, 4, 8]
  UMAP runs: 2 weight combos × 3 clusters = 6
  HDBSCAN combos per UMAP: 9  |  Total combinations: 18


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


KeyboardInterrupt: 

In [10]:
# [RUN MESO] cluster within each macro group and save

import importlib
from config import cluster_config as cfg
importlib.reload(cfg)

meso_records = []

for macro_id in tqdm(sorted(macro_assignments["macro_cluster"].dropna().unique())):
    parent_ids  = macro_assignments[macro_assignments["macro_cluster"] == macro_id]["id"].values
    n_neighbors = proportional_min_cluster_size(
        len(parent_ids), cfg.MESO_N_NEIGHBORS_FRACTION,
        cfg.MESO_N_NEIGHBORS_MIN, cfg.MESO_N_NEIGHBORS_MAX,
    )
    min_cls = proportional_min_cluster_size(
        len(parent_ids), cfg.MESO_MIN_CLUSTER_SIZE_FRACTION,
        cfg.MESO_MIN_CLUSTER_SIZE_MIN, cfg.MESO_MIN_CLUSTER_SIZE_MAX,
    )
    labels, subset_ids = cluster_within(
        matrix, embeddings, ids, embedding_ids,
        parent_ids=parent_ids,
        semantic_weight=cfg.MESO_SEMANTIC_WEIGHT,
        n_neighbors=n_neighbors,
        n_components=cfg.UMAP_N_COMPONENTS,
        min_cluster_size=min_cls,
        min_samples=cfg.MESO_MIN_SAMPLES,
        slot_embeddings=slot_embeddings,
        slot_embedding_ids=slot_embedding_ids,
        slot_weight=cfg.MESO_SLOT_WEIGHT,
    )
    for aid, label in zip(subset_ids, labels):
        meso_id = int(macro_id) * 1000 + int(label) if label >= 0 else np.nan
        meso_records.append({"id": str(aid), "meso_cluster": meso_id})

meso_assignments = pd.DataFrame(meso_records)
meso_assignments.to_csv(cfg.MESO_ASSIGNMENTS_CSV, index=False)

n_meso     = meso_assignments["meso_cluster"].nunique()
noise_rate = meso_assignments["meso_cluster"].isna().mean()
print(f"Meso clusters: {n_meso}")
print(f"Noise rate:    {noise_rate:.1%}")
print(f"Saved → {cfg.MESO_ASSIGNMENTS_CSV}")

  0%|          | 0/17 [00:00<?, ?it/s]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
  6%|▌         | 1/17 [00:40<10:48, 40.51s/it]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
 12%|█▏        | 2/17 [00:54<06:14, 24.97s/it]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
 18%|█▊        | 3/17 [00:58<03:35, 15.38s/it]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
 24%|██▎       | 4/17 [01:07<02:45, 12.7

Meso clusters: 226
Noise rate:    22.8%
Saved → data/cluster_assignments_meso.csv


In [11]:
# [VALIDATE MESO] coherence check — flags catch-all and tiny clusters

meso_val = validate_clustering(
    assignments=meso_assignments,
    cluster_col="meso_cluster",
    extractions=extractions,
    catchall_threshold=cfg.COHERENCE_CATCHALL_THRESHOLD,
    min_size_warning=cfg.COHERENCE_MIN_SIZE_WARNING,
    target_noise_max=cfg.COHERENCE_TARGET_NOISE_MAX,
    target_coherence_min=cfg.COHERENCE_TARGET_COHERENCE,
    target_catchall_max=cfg.COHERENCE_TARGET_CATCHALL_MAX,
)
meso_val.head(30)


VALIDATION — MESO_CLUSTER
  Clusters:        226
  Noise rate:      22.8%  (target < 25%)
  Avg coherence:   46.6%  (top10_cumul, target > 30%)
  Catch-all rate:  23%  (target < 40%)
  Tiny clusters:   29  (< 25 articles)

  VERDICT: ✓ PASS

  Flagged clusters (worst first):
 cluster  n_articles  top10_cumul     flags
    7010         333         0.08 CATCH-ALL
   13012        1171         0.09 CATCH-ALL
    3004         279         0.10 CATCH-ALL
   14009         145         0.11 CATCH-ALL
    7004         360         0.12 CATCH-ALL
    7015         118         0.12 CATCH-ALL
    2009          92         0.14 CATCH-ALL
   13008         179         0.14 CATCH-ALL
   13014         290         0.14 CATCH-ALL
    3005         570         0.15 CATCH-ALL
    5002         547         0.16 CATCH-ALL
   11003         144         0.16 CATCH-ALL
    7024         115         0.16 CATCH-ALL
    7023         127         0.16 CATCH-ALL
   16012          98         0.17 CATCH-ALL
    7009          7

,cluster,n_articles,n_uniq_subj,top1_pct,top10_cumul,flags
107,7010,333,347,0.01,0.08,CATCH-ALL
172,13012,1171,1092,0.02,0.09,CATCH-ALL
41,3004,279,320,0.03,0.10,CATCH-ALL
186,14009,145,162,0.02,0.11,CATCH-ALL
101,7004,360,313,0.02,0.12,CATCH-ALL
112,7015,118,133,0.03,0.12,CATCH-ALL
30,2009,92,98,0.02,0.14,CATCH-ALL
168,13008,179,199,0.02,0.14,CATCH-ALL
174,13014,290,257,0.02,0.14,CATCH-ALL
42,3005,570,479,0.06,0.15,CATCH-ALL


## Micro Clustering

In [15]:
# [TUNE MICRO] grid search over slot_weight, n_neighbors_fraction, mcs_fraction, min_samples
# Fixed: semantic_weight=0.05 (known good), umap_n_components, umap_min_dist, umap_metric, mcs_min, mcs_max
# UMAP runs once per (slot_weight, n_neighbors_fraction, parent_cluster)
# Skip if parameters are already confirmed

micro_tune_df = tune_within(
    matrix=matrix, embeddings=embeddings, ids=ids, embedding_ids=embedding_ids,
    slot_embeddings=slot_embeddings, slot_embedding_ids=slot_embedding_ids,
    extractions=extractions,
    parent_assignments=meso_assignments,
    parent_col="meso_cluster",
    semantic_weights=cfg.TUNE_MICRO_SEMANTIC_WEIGHTS,
    slot_weights=cfg.TUNE_MICRO_SLOT_WEIGHTS,
    n_neighbors_fractions=cfg.TUNE_MICRO_N_NEIGHBORS_FRACS,
    mcs_fractions=cfg.TUNE_MICRO_MCS_FRACTIONS,
    mcs_min=cfg.MICRO_MIN_CLUSTER_SIZE_MIN,
    mcs_max=cfg.MICRO_MIN_CLUSTER_SIZE_MAX,
    min_samples_values=cfg.TUNE_MICRO_MIN_SAMPLES,
    n_neighbors_min=cfg.MICRO_N_NEIGHBORS_MIN,
    n_neighbors_max=cfg.MICRO_N_NEIGHBORS_MAX,
    umap_n_components=cfg.UMAP_N_COMPONENTS,
    umap_min_dist=cfg.UMAP_MIN_DIST,
    umap_metric=cfg.UMAP_METRIC,
    hdbscan_metric=cfg.HDBSCAN_METRIC,
    target_noise_max=cfg.COHERENCE_TARGET_NOISE_MAX,
    catchall_threshold=cfg.COHERENCE_CATCHALL_THRESHOLD,
)

print(micro_tune_df.head(20).to_string(index=False))

rec = micro_tune_df.iloc[0]
print(f"\nRecommended config:")
print(f"  MICRO_SLOT_WEIGHT               = {rec['slot_weight']}")
print(f"  MICRO_N_NEIGHBORS_FRACTION      = {rec['n_neighbors_frac']}")
print(f"  MICRO_MIN_CLUSTER_SIZE_FRACTION = {rec['mcs_fraction']}")
print(f"  MICRO_MIN_SAMPLES               = {int(rec['min_samples'])}")

  Tuning on 3 parent clusters: [5004, 10008, 4000]
  UMAP runs: 12 weight combos × 3 clusters = 36
  HDBSCAN combos per UMAP: 6  |  Total combinations: 72


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  sw=0.05 slw=0.1 nb_frac=0.03 — UMAP done, running HDBSCAN grid...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  sw=0.05 slw=0.1 nb_frac=0.05 — UMAP done, running HDBSCAN grid...


...

  sw=0.05 slw=0.4 nb_frac=0.03 — UMAP done, running HDBSCAN grid...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  sw=0.05 slw=0.4 nb_frac=0.05 — UMAP done, running HDBSCAN grid...


c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  sw=0.05 slw=0.4 nb_frac=0.08 — UMAP done, running HDBSCAN grid...
 RANK  semantic_weight  slot_weight  n_neighbors_frac  mcs_fraction  min_samples  avg_n_clusters  avg_noise_rate  avg_top10_cumul  catchall_rate  noise_penalty  score
    1             0.05          0.1              0.03          0.03            3             6.0           0.129            0.824          0.167            0.0  0.740
    2             0.05          0.1              0.03          0.05            3             6.0           0.129            0.824          0.167            0.0  0.740
    3             0.05          0.1              0.05          0.03            3             6.0           0.129            0.824          0.167            0.0  0.740
    4             0.05          0.1              0.03          0.08            3             6.0           0.129            0.824          0.167            0.0  0.740
    5             0.05          0.1              0.08          0.03            3             6.0 

In [12]:
# [RUN MICRO] cluster within each meso group and save

import importlib
from config import cluster_config as cfg
importlib.reload(cfg)

micro_records = []

for meso_id in tqdm(sorted(meso_assignments["meso_cluster"].dropna().unique())):
    parent_ids  = meso_assignments[meso_assignments["meso_cluster"] == meso_id]["id"].values
    n_neighbors = proportional_min_cluster_size(
        len(parent_ids), cfg.MICRO_N_NEIGHBORS_FRACTION,
        cfg.MICRO_N_NEIGHBORS_MIN, cfg.MICRO_N_NEIGHBORS_MAX,
    )
    min_cls = proportional_min_cluster_size(
        len(parent_ids), cfg.MICRO_MIN_CLUSTER_SIZE_FRACTION,
        cfg.MICRO_MIN_CLUSTER_SIZE_MIN, cfg.MICRO_MIN_CLUSTER_SIZE_MAX,
    )
    labels, subset_ids = cluster_within(
        matrix, embeddings, ids, embedding_ids,
        parent_ids=parent_ids,
        semantic_weight=cfg.MICRO_SEMANTIC_WEIGHT,
        n_neighbors=n_neighbors,
        n_components=cfg.UMAP_N_COMPONENTS,
        min_cluster_size=min_cls,
        min_samples=cfg.MICRO_MIN_SAMPLES,
        slot_embeddings=slot_embeddings,
        slot_embedding_ids=slot_embedding_ids,
        slot_weight=cfg.MICRO_SLOT_WEIGHT,
    )
    for aid, label in zip(subset_ids, labels):
        micro_id = int(meso_id) * 1000 + int(label) if label >= 0 else np.nan
        micro_records.append({"id": str(aid), "micro_cluster": micro_id})

micro_assignments = pd.DataFrame(micro_records)
micro_assignments.to_csv(cfg.MICRO_ASSIGNMENTS_CSV, index=False)

n_micro    = micro_assignments["micro_cluster"].nunique()
noise_rate = micro_assignments["micro_cluster"].isna().mean()
print(f"Micro clusters: {n_micro}")
print(f"Noise rate:     {noise_rate:.1%}")
print(f"Saved → {cfg.MICRO_ASSIGNMENTS_CSV}")

  0%|          | 0/226 [00:00<?, ?it/s]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
  0%|          | 1/226 [00:01<05:20,  1.43s/it]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
  1%|          | 2/226 [01:19<2:53:24, 46.45s/it]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
  1%|▏         | 3/226 [01:19<1:34:07, 25.33s/it]c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
  2%|▏         | 4/226 [01:19<57

Micro clusters: 351
Noise rate:     24.5%
Saved → data/cluster_assignments_micro.csv


In [13]:
# [VALIDATE MICRO] coherence check — flags catch-all and tiny clusters

micro_val = validate_clustering(
    assignments=micro_assignments,
    cluster_col="micro_cluster",
    extractions=extractions,
    catchall_threshold=cfg.COHERENCE_CATCHALL_THRESHOLD,
    min_size_warning=cfg.COHERENCE_MIN_SIZE_WARNING,
    target_noise_max=cfg.COHERENCE_TARGET_NOISE_MAX,
    target_coherence_min=cfg.COHERENCE_TARGET_COHERENCE,
    target_catchall_max=cfg.COHERENCE_TARGET_CATCHALL_MAX,
)
micro_val.head(30)


VALIDATION — MICRO_CLUSTER
  Clusters:        351
  Noise rate:      24.5%  (target < 25%)
  Avg coherence:   52.0%  (top10_cumul, target > 30%)
  Catch-all rate:  15%  (target < 40%)
  Tiny clusters:   58  (< 25 articles)

  VERDICT: ✓ PASS

  Flagged clusters (worst first):
 cluster  n_articles  top10_cumul     flags
13012000         278         0.09 CATCH-ALL
13012001         869         0.10 CATCH-ALL
 3004003          77         0.16 CATCH-ALL
 7012000         168         0.16 CATCH-ALL
 7015001          77         0.17 CATCH-ALL
 3005004          71         0.17 CATCH-ALL
 3004001          73         0.18 CATCH-ALL
 7010005          57         0.18 CATCH-ALL
 7004004          67         0.19 CATCH-ALL
 7016001          63         0.19 CATCH-ALL
 7012005         123         0.19 CATCH-ALL
 7004001         132         0.20 CATCH-ALL
 7023001          59         0.20 CATCH-ALL
13005000          93         0.20 CATCH-ALL
 3024001         117         0.20 CATCH-ALL
 7010003         1

,cluster,n_articles,n_uniq_subj,top1_pct,top10_cumul,flags
290,13012000,278,296,0.01,0.09,CATCH-ALL
291,13012001,869,817,0.02,0.10,CATCH-ALL
91,3004003,77,96,0.03,0.16,CATCH-ALL
183,7012000,168,205,0.02,0.16,CATCH-ALL
194,7015001,77,86,0.04,0.17,CATCH-ALL
96,3005004,71,78,0.03,0.17,CATCH-ALL
89,3004001,73,97,0.03,0.18,CATCH-ALL
182,7010005,57,63,0.02,0.18,CATCH-ALL
172,7004004,67,68,0.03,0.19,CATCH-ALL
196,7016001,63,81,0.03,0.19,CATCH-ALL


In [14]:
# how many micro clusters does each meso cluster have?
micro_per_meso = (
    micro_assignments.dropna(subset=["micro_cluster"])
    .assign(meso_parent=lambda d: (d["micro_cluster"] // 1000).astype(int))
    .groupby("meso_parent")["micro_cluster"].nunique()
)

# include meso clusters with zero micro children (all noise)
all_meso = meso_assignments["meso_cluster"].dropna().astype(int).unique()
micro_counts = pd.Series(0, index=all_meso)
micro_counts.update(micro_per_meso)

print("Micro clusters per meso cluster:")
print(micro_counts.value_counts().sort_index())
print(f"\nMeso clusters with 0 micro: {(micro_counts == 0).sum()}")
print(f"Meso clusters with 1 micro: {(micro_counts == 1).sum()}")
print(f"Meso clusters with 2+ micro: {(micro_counts >= 2).sum()}")

Micro clusters per meso cluster:
0     114
2      58
3      24
4       9
5      12
6       4
8       4
11      1
Name: count, dtype: int64

Meso clusters with 0 micro: 114
Meso clusters with 1 micro: 0
Meso clusters with 2+ micro: 112


In [15]:
zero_micro_mesos = micro_counts[micro_counts == 0].index
meso_sizes = meso_assignments["meso_cluster"].dropna().astype(int).value_counts()

zero_sizes = meso_sizes[meso_sizes.index.isin(zero_micro_mesos)]
print(f"Meso clusters with 0 micro children: {len(zero_sizes)}")
print(f"  Size range: {zero_sizes.min()}–{zero_sizes.max()}")
print(f"  Median size: {zero_sizes.median():.0f}")
print(f"  Total articles in them: {zero_sizes.sum():,}")
print(f"  Largest 10:\n{zero_sizes.sort_values(ascending=False).head(10)}")

Meso clusters with 0 micro children: 114
  Size range: 15–121
  Median size: 36
  Total articles in them: 4,390
  Largest 10:
meso_cluster
3028     121
10000     86
14010     85
9009      78
1007      75
3007      74
16000     74
3029      72
7008      62
16016     62
Name: count, dtype: int64


## Save and Profile

In [16]:
# save combined assignments and profile noise at each level

all_assignments = (
    macro_assignments
    .merge(meso_assignments,  on="id", how="left")
    .merge(micro_assignments, on="id", how="left")
)
all_assignments.to_csv(cfg.ASSIGNMENTS_CSV, index=False)
print(f"Saved combined assignments → {cfg.ASSIGNMENTS_CSV}")
print(f"  Rows: {len(all_assignments):,}")
print(f"  Macro clusters: {all_assignments['macro_cluster'].nunique()}")
print(f"  Meso clusters:  {all_assignments['meso_cluster'].nunique()}")
print(f"  Micro clusters: {all_assignments['micro_cluster'].nunique()}")

Saved combined assignments → data/cluster_assignments.csv
  Rows: 44,876
  Macro clusters: 17
  Meso clusters:  226
  Micro clusters: 351


In [17]:
# profile noise — sample articles that didn't fit any cluster at a given level
# change cluster_col to inspect different levels

cluster_col = "micro_cluster"   # macro_cluster | meso_cluster | micro_cluster

joined = join_extractions(all_assignments, extractions)
pd.set_option("display.max_colwidth", None)
profile_noise(joined, cluster_col=cluster_col, n_examples=10)

Noise articles at micro_cluster level: 20,876 (46.5%)


,id,dominant_narrative
291888,60370603,"The SVP pursues a reorientation of the VBS (Department of Defence) away from what it deems peripheral activities toward military core functions, seeking to free resources for defence spending without tax increases, against Minister Pfister's preferred strategy of revenue expansion."
276116,58920136,"Stadler Rail, responding to rising US demand and a Swiss-US tariff deal, expands its Salt Lake City manufacturing facility by adding 200 jobs and relocating welding operations from Hungary to the United States."
2846,57822037,"OpenAI pursues trademark protection and enforcement of the term 'GPT' in Switzerland against AlpineAI's competing product SwissGPT, while AlpineAI resists the claim and prepares to defend its market position through legal challenge."
331878,57207497,"Christoph Mörgeli and Stefan Müller-Altermatt pursue accountability and reputational defence of the Swiss intelligence service (NDB) against what they characterise as suggestive, politically motivated reporting by SRF Investigativ that amplifies unsubstantiated allegations of improper Russian intelligence collaboration."
300961,59584482,"The European Union and India pursue a comprehensive trade agreement to reduce tariffs and diversify their economic partnerships away from dependence on the United States and China, overcoming years of stalled negotiations through accelerated six-month talks."
78563,58283550,"Brazilian police pursue the dismantling of illegal wildlife trafficking networks by conducting a major enforcement operation that rescues over 700 animals and arrests suspects, confronting the structural challenge of controlling a vast territory with deeply entrenched criminal syndicates."
278048,58185962,"The National Council pursues a compromise weakening of the right of association to lodge complaints against 16 hydropower projects, against the Council of States' push to abolish it entirely, with the outcome pending a second Council of States vote."
152375,59495363,"US President Trump pursues the annexation of Greenland by threatening military force and economic sanctions, motivated partly by anticipated climate-driven increases in Arctic resources and strategic value, while eight European NATO allies resist his territorial claims and defend Greenland's sovereignty."
370377,58480359,"Farmer Benno Trütsch pursues renewal of his land lease with Kloster Einsiedeln by threatening to close the cross-country ski trail that runs across his rented land, against the monastery's decision to terminate his lease in 2026 and redirect the land to resident farmers."
265411,58481003,"Christiane H. pursues the creation and expansion of a white-supremacist dating platform and broader organisational infrastructure to unite far-right actors across borders around the conspiracy theory of 'white genocide,' overcoming initial personal ideological conversion and building networks with established neo-Nazi figures and organisations."
